# Path B Sweep: CMA + raw KM events / 60Hz telem stream (state + trend)

1 task × 4 modality combos × 2 splits × 3 seeds = **24 runs**

| Field | Value |
|-------|-------|
| Sweep | `configs/sweeps/pathB_cma_state_trend.yaml` |
| Task | `cma_pathB_state_trend` (multitask classification: state + trend) |
| Fusion | CMA (anchor=video, cm_layers=4, sa_layers=2) |
| Modalities | `[video, km_event, telem_60hz]`, `[video, km_event]`, `[video, telem_60hz]`, `[km_event, telem_60hz]` |
| Features | `video_clip` (CLIP), `km_event` (raw event tokens), `telem_60hz` (60Hz stream) |
| Splits | cross_subject, within_subject |
| Patience | 8 |

**Pre-req**: `features/aligned/{video_clip,km_event,telem_60hz}` 已生成（由 `encoder.common.extract_km_event` 与 `encoder.common.extract_telem_60hz`）。

Disconnect-safe: re-run Cell 3 to resume from where it left off.

In [2]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
/content/drive/MyDrive/ProjectExperiment
NVIDIA A100-SXM4-80GB, 81920 MiB


In [3]:
# Cell 2: Dry run — preview plan, verify no import errors
!python scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_trend.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend \
    --dry_run 2>&1 | tail -10

!echo "---"
!python scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_trend.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend \
    --dry_run 2>&1 | grep -c '\[RUN\]'

  [RUN] cma_pathB_state_trend / triple_video_km_event_telem_60hz / seed2
  [RUN] cma_pathB_state_trend / dual_video_km_event / seed0
  [RUN] cma_pathB_state_trend / dual_video_km_event / seed1
  [RUN] cma_pathB_state_trend / dual_video_km_event / seed2
  [RUN] cma_pathB_state_trend / dual_video_telem_60hz / seed0
  [RUN] cma_pathB_state_trend / dual_video_telem_60hz / seed1
  [RUN] cma_pathB_state_trend / dual_video_telem_60hz / seed2
  [RUN] cma_pathB_state_trend / dual_km_event_telem_60hz / seed0
  [RUN] cma_pathB_state_trend / dual_km_event_telem_60hz / seed1
  [RUN] cma_pathB_state_trend / dual_km_event_telem_60hz / seed2
---
24


In [5]:
# Cell 3: Run experiments (re-run this cell to resume after disconnect)
# -u: unbuffered stdout so per-run progress prints in real time
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathB_cma_state_trend.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend

Sweep plan: 24 runs total, 23 already completed, 1 to run

[1/24] SKIP cma_pathB_state_trend/triple_video_km_event_telem_60hz/seed0 (already done)
[2/24] SKIP cma_pathB_state_trend/triple_video_km_event_telem_60hz/seed1 (already done)

[3/24] [cross_subject] cma_pathB_state_trend / triple_video_km_event_telem_60hz / seed2
/content/drive/MyDrive/ProjectExperiment/src/models/encoders/km/event_token.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
Model parameters: 16,167,430 (trainable: 16,167,430)
Train samples: 605, Val samples: 204
Run directory: /content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend/cross_subject/cma_pathB_state_trend_3seed/triple_video_km_event_telem_60hz/2026-04-19_17-03-13__amucs_seq_mixed_rate__cma__km_event_telem_60hz_video__seed2
Epoch 1/40 | train_balanced_acc_mean: 0.3345 | train_balanced_acc_s

In [6]:
# Cell 4: Check progress (run anytime)
import glob
from collections import Counter

RUNS_ROOT = '/content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend'
TOTAL = 24  # 1 task x 4 modality combos x 2 splits x 3 seeds

done = glob.glob(f'{RUNS_ROOT}/**/metrics.json', recursive=True)
print(f'Completed: {len(done)} / {TOTAL}\n')

# Per split + modality breakdown
buckets = Counter()
for p in done:
    parts = p.replace('\\', '/').split('/')
    split = 'unknown'
    mod_dir = 'unknown'
    for i, part in enumerate(parts):
        if part in ('cross_subject', 'within_subject'):
            split = part
        if part.startswith(('single_', 'dual_', 'triple_')):
            mod_dir = part
    buckets[f'{split}/{mod_dir}'] += 1

expected_per_combo = 3  # 3 seeds
for key, count in sorted(buckets.items()):
    print(f'  {key}: {count}/{expected_per_combo}')

Completed: 23 / 24

  cross_subject/dual_km_event_telem_60hz: 3/3
  cross_subject/dual_video_km_event: 3/3
  cross_subject/dual_video_telem_60hz: 3/3
  cross_subject/triple_video_km_event_telem_60hz: 2/3
  within_subject/dual_km_event_telem_60hz: 3/3
  within_subject/dual_video_km_event: 3/3
  within_subject/dual_video_telem_60hz: 3/3
  within_subject/triple_video_km_event_telem_60hz: 3/3


In [7]:
# Cell 5: Quick results summary across modality combos x splits
import json, glob, numpy as np
from pathlib import Path
from collections import defaultdict

RUNS_ROOT = Path('/content/drive/MyDrive/AmuCS_experiment/runs/pathB_cma_state_trend')
TASK_DIR = 'cma_pathB_state_trend_3seed'

mod_dirs = {
    'V+K+T': 'triple_video_km_event_telem_60hz',
    'V+K':   'dual_video_km_event',
    'V+T':   'dual_video_telem_60hz',
    'K+T':   'dual_km_event_telem_60hz',
}

def collect(split, mod_dir):
    pattern = f'{RUNS_ROOT}/{split}/{TASK_DIR}/{mod_dir}/**/metrics.json'
    out = []
    for f in sorted(glob.glob(pattern, recursive=True)):
        with open(f) as fp:
            out.append(json.load(fp))
    return out

def mean_of(metrics_list, key):
    vals = [m[key] for m in metrics_list if key in m and isinstance(m[key], (int, float))]
    return f'{np.mean(vals):.4f}' if vals else '  N/A '

# Auto-discover test_* keys to handle multitask schema (state/trend per-task + mean)
all_keys = set()
for split in ['cross_subject', 'within_subject']:
    for mod_dir in mod_dirs.values():
        for m in collect(split, mod_dir):
            all_keys.update(k for k in m if k.startswith('test_') and isinstance(m[k], (int, float)))

preferred = ['test_balanced_acc_mean', 'test_f1_mean',
             'test_state_balanced_acc', 'test_state_macro_f1',
             'test_trend_balanced_acc', 'test_trend_macro_f1']
show_keys = [k for k in preferred if k in all_keys] or sorted(all_keys)

for split in ['cross_subject', 'within_subject']:
    print(f'\n{"="*90}')
    print(f'  {split}')
    print(f'{"="*90}')
    header = f'{"Modalities":<10}' + ''.join(f'  {k:>22}' for k in show_keys)
    print(header)
    print('-' * len(header))
    for mod_label, mod_dir in mod_dirs.items():
        ms = collect(split, mod_dir)
        row = f'{mod_label:<10}'
        for k in show_keys:
            row += f'  {mean_of(ms, k):>22}'
        print(row)


  cross_subject
Modalities  test_balanced_acc_mean            test_f1_mean
----------------------------------------------------------
V+K+T                       0.4113                  0.3906
V+K                         0.4419                  0.4232
V+T                         0.4144                  0.3931
K+T                         0.3676                  0.3540

  within_subject
Modalities  test_balanced_acc_mean            test_f1_mean
----------------------------------------------------------
V+K+T                       0.3541                  0.3370
V+K                         0.3589                  0.3203
V+T                         0.3426                  0.3239
K+T                         0.3496                  0.3241
